# Deep Learning for Sequence Modeling and Recurrent Neural Networks (RNN)

# 1. Introduction to Sequence Modeling

Sequence modeling focuses on data where order matters.

Examples:

* Text
* Speech
* Time series
* DNA sequences
* User activity logs
* Stock prices
* Sensor measurements

Unlike classical ML problems:

```text
x1, x2, x3 -> y
```

sequence problems involve:

```text
x1 -> x2 -> x3 -> x4 -> ...
```

where previous values influence future values.

---

**Common Sequence Problems**

| Problem                 | Example                      |
| ----------------------- | ---------------------------- |
| Next word prediction    | "I love deep ___"            |
| Sentiment analysis      | Positive/negative review     |
| Machine translation     | English -> French            |
| Time series forecasting | Predict tomorrow temperature |
| Speech recognition      | Audio -> Text                |
| Video understanding     | Action classification        |

---

**Why Sequence Modeling Matters**

Traditional models assume:

* independent samples
* fixed-size inputs
* no temporal dependency

But real-world sequential data contains:

* memory
* context
* temporal dependencies
* ordering information

RNNs were designed to process such dependencies.

5 real-world applications - TODO:

# 2. Why Feed-Forward Neural Networks Fail on Sequential Data

A Feed-Forward Neural Network (FFNN) processes each input independently.

Architecture:

```text
Input -> Hidden -> Output
```

Problems with sequences:

* no memory
* fixed input size
* ignores temporal dependencies
* cannot share information across time

---

**Example Problem**

Sentence:

```text
The movie was not good
```

A simple FFNN may fail to understand:

* "not" changes meaning of "good"
* word ordering matters

FFNN treats features independently.

RNN introduces:

* hidden state
* memory over time
* parameter sharing across sequence steps

---

## FFNN vs RNN

| Feature                  | FFNN      | RNN     |
| ------------------------ | --------- | ------- |
| Memory                   | No        | Yes     |
| Sequential understanding | Weak      | Strong  |
| Shared weights over time | No        | Yes     |
| Variable sequence length | Difficult | Natural |
| Temporal dependencies    | No        | Yes     |

---

TODO:

Why would FFNN struggle with:

Machine translation
Speech recognition
Stock prediction

# 3. Recurrent Neural Networks (RNN) Basics


RNN introduces a hidden state that carries information from previous steps.

At time step t:

* input: x_t
* previous hidden state: h_(t-1)
* current hidden state: h_t
* output: y_t

---

**RNN Computation**

The hidden state is updated recursively at each time step.

**Hidden State Update**

$
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)
$

**Output Computation**

$
y_t = W_{hy}h_t + b_y
$

Where:

* (x_t) = input at time step (t)
* (h_t) = hidden state at time step (t)
* (h_{t-1}) = previous hidden state
* (y_t) = output at time step (t)
* (W_{xh}) = input-to-hidden weights
* (W_{hh}) = hidden-to-hidden weights
* (W_{hy}) = hidden-to-output weights
* (b_h), (b_y) = bias vectors


At each time step, the RNN:

1. Takes the current input (x_t)
2. Combines it with memory from the previous hidden state (h_{t-1})
3. Produces a new hidden state (h_t)
4. Generates output (y_t)

This recursive structure allows RNNs to remember information from earlier parts of the sequence.

---

**Simplified Flow**

```text
Previous Memory (h_t-1)
          ↓
Current Input (x_t)
          ↓
   RNN Cell Computation
          ↓
 New Hidden State (h_t)
          ↓
      Output (y_t)
```

**Intuition**

The hidden state acts like memory.

Example:

```text
I grew up in France and I speak fluent ____
```

RNN remembers:

* France
* language context

and predicts:

```text
French
```

---

**Unrolled RNN**

Instead of one repeated block:

```text
RNN Cell
```

we can visualize:

```text
x1 -> h1 -> y1
x2 -> h2 -> y2
x3 -> h3 -> y3
```

Each step shares the SAME weights.

This is extremely important.

---

# 4. RNN Forward Pass Mathematics

In [100]:
import numpy as np

x = np.array([1.0, 0.5])
h_prev = np.array([0.1, -0.2])

In [101]:
# Weights:
np.random.seed(1)
Wxh = np.random.randn(2, 2)
Whh = np.random.randn(2, 2)
Why = np.random.randn(1, 2)

In [102]:
Wxh, Whh, Why

(array([[ 1.62434536, -0.61175641],
        [-0.52817175, -1.07296862]]),
 array([[ 0.86540763, -2.3015387 ],
        [ 1.74481176, -0.7612069 ]]),
 array([[ 0.3190391 , -0.24937038]]))

In [103]:
# Manual Forward Pass
h = np.tanh(Wxh @ x + Whh @ h_prev)
y = Why @ h

print('Hidden state:', h)
print('Output:', y)

Hidden state: [ 0.95316755 -0.62789501]
Output: [0.46067613]



**Understanding Shapes**

| Tensor | Meaning                  |
| ------ | ------------------------ |
| x_t    | input vector             |
| h_t    | hidden state             |
| Wxh    | input -> hidden weights  |
| Whh    | hidden -> hidden weights |
| Why    | hidden -> output weights |

# 5. Backpropagation Through Time (BPTT)

RNNs are trained using:

* gradient descent
* Backpropagation Through Time (BPTT)

Unlike Feed-Forward Neural Networks, RNNs reuse the **same weights** across all sequence steps.

Because of this, the computational graph extends through time:

```text id="h4d0ap"
x1 → h1 → h2 → h3 → h4 → y
```

During training, gradients must flow backward through all previous hidden states and time steps.

---

**Key Idea**

The prediction at time step (t) depends not only on the current input, but also on previous hidden states.

This means:

* earlier states influence future outputs
* loss accumulates across the sequence
* gradients must propagate through time

---

**Chain Rule Across Time**

The total gradient with respect to RNN weights is accumulated across all time steps:

$$
\frac{\partial L}{\partial W}=
\sum_t
\frac{\partial L_t}{\partial h_t}
\frac{\partial h_t}{\partial W}
$$

Where:

* (L) = total sequence loss
* (L_t) = loss at time step (t)
* (h_t) = hidden state at time step (t)
* (W) = shared RNN weights

Because the same weights are reused repeatedly, gradients are multiplied many times during backpropagation.

---

## Why Training Becomes Difficult

Repeated gradient multiplication can make gradients:

* extremely small
* extremely large

### Vanishing Gradients

Small values repeatedly multiplied shrink toward zero.

Example:

```text id="m5s8xw"
0.9 × 0.9 × 0.9 × ...
→ 0
```

Result:

* long-term dependencies are forgotten
* early sequence information disappears
* training becomes slow or ineffective

Example problem:

```text id="k9v2dy"
Trying to remember a word from
200 tokens earlier in a sentence
```

---

### Exploding Gradients

Large values repeatedly multiplied grow uncontrollably.

Example:

```text id="x8n6tp"
1.5 × 1.5 × 1.5 × ...
→ very large values
```

Result:

* unstable updates
* oscillating loss
* NaN values
* training divergence

---

**Visual Intuition**

```text id="r7f4la"
Forward pass:
x1 → h1 → h2 → h3 → h4 → y

Backward pass:
y → h4 → h3 → h2 → h1
```

The longer the sequence, the deeper the effective network becomes.

---

**Common Solutions**

| Problem                | Solution                 |
| ---------------------- | ------------------------ |
| Vanishing gradients    | LSTM / GRU               |
| Exploding gradients    | Gradient clipping        |
| Very long dependencies | Attention / Transformers |

---

**Gradient Clipping Example**

```python id="p2j7zc"
torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)
```

This limits gradient magnitude and stabilizes training.

# 6. Implementing RNN From Scratch with NumPy

In [104]:
# Dataset
text = 'hello world'
chars = sorted(list(set(text)))
chars

[' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']

In [105]:
char_to_ix = {ch:i for i,ch in enumerate(chars)}
ix_to_char = {i:ch for ch,i in char_to_ix.items()}
char_to_ix, ix_to_char

({' ': 0, 'd': 1, 'e': 2, 'h': 3, 'l': 4, 'o': 5, 'r': 6, 'w': 7},
 {0: ' ', 1: 'd', 2: 'e', 3: 'h', 4: 'l', 5: 'o', 6: 'r', 7: 'w'})

In [106]:
# Initialize Parameters
import numpy as np

np.random.seed(1)

vocab_size = len(chars)
hidden_size = 16

Wxh = np.random.randn(hidden_size, vocab_size) * 0.01
Whh = np.random.randn(hidden_size, hidden_size) * 0.01
Why = np.random.randn(vocab_size, hidden_size) * 0.01

bh = np.zeros((hidden_size, 1))
by = np.zeros((vocab_size, 1))

In [107]:
# Forward Pass
h_prev = np.zeros((hidden_size, 1))

inputs = [char_to_ix[ch] for ch in 'hello']

for t in range(len(inputs)):
    x = np.zeros((vocab_size, 1))
    x[inputs[t]] = 1

    h_prev = np.tanh(Wxh @ x + Whh @ h_prev + bh)
    y = Why @ h_prev + by

    print(y.ravel())

[ 1.04221688e-04  1.52727244e-05  6.79070819e-04  9.41841087e-04
 -8.89940915e-04 -8.66316038e-04  5.96896008e-04 -2.22645223e-05]
[ 5.77364896e-04 -6.65972666e-05  4.74614678e-04  2.29664750e-04
  1.02298193e-04 -1.32789994e-04 -1.84144187e-04 -1.42348848e-04]
[-0.00028885  0.00040627  0.00028043 -0.00039833 -0.00045808 -0.00011075
  0.00014251  0.00043354]
[-0.00027133  0.0003943   0.00027848 -0.00041034 -0.00044943 -0.00011989
  0.00011158  0.00042963]
[ 2.29946351e-04  6.24085069e-04 -4.92226485e-04  1.37439358e-05
  1.29359858e-05 -2.00505503e-05 -1.24498956e-04 -5.04995110e-04]


## Softmax

Softmax converts raw output scores (**logits**) into probabilities.

$
P_i = \frac{e^{x_i}}{\sum_j e^{x_j}}
$

In [108]:
def softmax(x):
    # numerical stability trick
    x = x - np.max(x)

    exp_x = np.exp(x)

    return exp_x / np.sum(exp_x)

In [109]:
sample = np.array([[2.0], [1.0], [0.1]])

probs = softmax(sample)

print(probs)
print("Sum:", np.sum(probs))

[[0.65900114]
 [0.24243297]
 [0.09856589]]
Sum: 1.0


## Generate Next Characters

Now we transform raw outputs into predicted characters.

### Convert Logits to Probabilities

Update the forward pass:

In [110]:
h_prev = np.zeros((hidden_size, 1))

inputs = [char_to_ix[ch] for ch in 'hello']

for t in range(len(inputs)):

    x = np.zeros((vocab_size, 1))
    x[inputs[t]] = 1

    # hidden state
    h_prev = np.tanh(Wxh @ x + Whh @ h_prev + bh)

    # logits
    y = Why @ h_prev + by

    # probabilities
    p = softmax(y)

    print(p.ravel())

[0.12500427 0.12499315 0.12507615 0.12510902 0.12488006 0.12488301
 0.12506587 0.12498846]
[0.12505877 0.12497827 0.12504592 0.1250153  0.12499938 0.12496999
 0.12496358 0.1249688 ]
[0.12496379 0.12505068 0.12503495 0.12495011 0.12494264 0.12498604
 0.1250177  0.12505409]
[0.12496666 0.12504987 0.12503539 0.12494929 0.1249444  0.12498559
 0.12501452 0.12505429]
[0.12503282 0.12508211 0.12494256 0.12500579 0.12500569 0.12500157
 0.12498851 0.12494096]


### Predict Next Character

Choose the character with highest probability.

In [111]:
predicted_index = np.argmax(p)

predicted_char = ix_to_char[predicted_index]

print("Predicted char:", predicted_char)

Predicted char: d


### Generate Multiple Characters

Now create a small text generator.

In [112]:
def gererate_text(current_char='h'):
    h = np.zeros((hidden_size, 1))

    generated = current_char

    for _ in range(20):

        x = np.zeros((vocab_size, 1))
        x[char_to_ix[current_char]] = 1

        h = np.tanh(Wxh @ x + Whh @ h + bh)

        y = Why @ h + by

        p = softmax(y)

        idx = np.argmax(p)

        current_char = ix_to_char[idx]

        generated += current_char

    return generated

print(gererate_text())

hhhhhhhhhhhhhhhhhhhhh


model is not trained yet

## Add Loss Computation

We use **cross-entropy loss**.

For classification:

$
L = -\log(P(\text{correct class}))
$

In [113]:
target_char = 'e'

target_idx = char_to_ix[target_char]

loss = -np.log(p[target_idx, 0])

print(loss)

2.0799011949101907


Now compute total loss for `"hello"`.

Targets are shifted by one character:

### Sequence Loss Implementation

In [115]:
h_prev = np.zeros((hidden_size, 1))

inputs = [char_to_ix[ch] for ch in 'hell']
targets = [char_to_ix[ch] for ch in 'ello']

loss = 0

for t in range(len(inputs)):

    x = np.zeros((vocab_size, 1))
    x[inputs[t]] = 1

    h_prev = np.tanh(Wxh @ x + Whh @ h_prev + bh)

    y = Why @ h_prev + by

    p = softmax(y)

    loss += -np.log(p[targets[t], 0])

print("Total loss:", loss)

Total loss: 8.317736444222753


## Implement Gradient Descent Updates

Now we train the network.

Goal:

* compute gradients
* update weights
* reduce loss

In [116]:
learning_rate = 0.1

# init gradient tensors
dWxh = np.zeros_like(Wxh)
dWhh = np.zeros_like(Whh)
dWhy = np.zeros_like(Why)

dbh = np.zeros_like(bh)
dby = np.zeros_like(by)


In [ ]:
learning_rate = 0.1

# Training loop over multiple passes through the dataset
for epoch in range(100):

    # Initialize hidden state at the beginning of each epoch
    # This represents "memory" of the RNN at time step 0
    h_prev = np.zeros((hidden_size, 1))

    # Track total loss over the sequence
    loss = 0

    # Loop over each character/time step in the input sequence
    for t in range(len(inputs)):

        # -------------------------
        # INPUT REPRESENTATION
        # -------------------------

        # Create one-hot encoded vector for current input character
        # Shape: (vocab_size, 1)
        x = np.zeros((vocab_size, 1))

        # Activate index corresponding to current character
        x[inputs[t]] = 1

        # -------------------------
        # FORWARD PASS (RNN CELL)
        # -------------------------

        # Compute new hidden state:
        # h_t = tanh(Wxh * x_t + Whh * h_(t-1) + b_h)
        #
        # - Wxh @ x  → contribution from current input
        # - Whh @ h_prev → contribution from previous memory
        # - bh → bias term
        #
        # tanh introduces non-linearity and keeps values bounded [-1, 1]
        h_prev = np.tanh(Wxh @ x + Whh @ h_prev + bh)

        # Compute raw output logits (unnormalized scores)
        # y = Why * h_t + by
        #
        # This maps hidden state → vocabulary space
        y = Why @ h_prev + by

        # Convert logits into probabilities using softmax
        # Now we interpret outputs as probability distribution over characters
        p = softmax(y)

        # -------------------------
        # LOSS COMPUTATION
        # -------------------------

        # Cross-entropy loss:
        # We penalize the model when it assigns low probability to the correct character
        #
        # loss = -log(P(correct_char))
        loss += -np.log(p[targets[t], 0])

        # -------------------------
        # BACKWARD PASS (GRADIENTS)
        # -------------------------

        # Gradient of softmax + cross entropy simplifies to:
        #
        # dy = predicted_probs
        # then subtract 1 at correct class index
        #
        # This gives direction to increase correct class probability
        dy = p.copy()
        dy[targets[t]] -= 1

        # -------------------------
        # GRADIENTS FOR OUTPUT LAYER
        # -------------------------

        # Gradient for Why:
        # dWhy = dL/dy * d(y)/d(Why)
        #
        # h_prev.T allows correct matrix alignment
        dWhy = dy @ h_prev.T

        # Bias gradient is just accumulated error signal
        dby = dy

        # -------------------------
        # BACKPROP THROUGH HIDDEN STATE
        # -------------------------

        # Propagate gradient back into hidden state
        # Why.T projects output error back into hidden space
        dh = Why.T @ dy

        # Derivative of tanh activation:
        # d(tanh(x)) = 1 - tanh(x)^2
        #
        # This adjusts gradient through non-linearity
        dh_raw = (1 - h_prev * h_prev) * dh

        # -------------------------
        # GRADIENTS FOR INPUT → HIDDEN WEIGHTS
        # -------------------------

        # Update how input affects hidden state
        dWxh = dh_raw @ x.T

        # Bias gradient for hidden layer
        dbh = dh_raw

        # -------------------------
        # PARAMETER UPDATE (GRADIENT DESCENT)
        # -------------------------

        # Adjust weights slightly in direction that reduces loss
        Wxh -= learning_rate * dWxh
        Why -= learning_rate * dWhy

        # Update biases
        bh -= learning_rate * dbh
        by -= learning_rate * dby

    # -------------------------
    # LOGGING
    # -------------------------

    # Print loss every 10 epochs to monitor training progress
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss:.4f}')

Epoch 0, Loss: 8.2958
Epoch 10, Loss: 5.9882
Epoch 20, Loss: 5.1221
Epoch 30, Loss: 4.0801
Epoch 40, Loss: 2.7947
Epoch 50, Loss: 2.3328
Epoch 60, Loss: 2.1048
Epoch 70, Loss: 1.9564
Epoch 80, Loss: 1.8816
Epoch 90, Loss: 1.8432


### Generate Multiple Characters after training

In [123]:
print(gererate_text())

helololololololololol


In [127]:
print(gererate_text(current_char='w'))
print(gererate_text(current_char='d'))

wlolololololololololo
dlolololololololololo


not very smart model :)

# 7. Building RNNs with PyTorch

Manual implementation is educational.

Frameworks provide:

* automatic differentiation
* GPU support
* optimized kernels
* easy experimentation


In [119]:
import torch
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, hidden = self.rnn(x)

        last_hidden = output[:, -1, :]

        return self.fc(last_hidden)

In [120]:
# Dummy Data
batch_size = 8
seq_length = 5
input_size = 10

x = torch.randn(batch_size, seq_length, input_size)

In [121]:
# Run Model
model = SimpleRNN(input_size=10, hidden_size=16, output_size=2)

out = model(x)

print(out.shape)

torch.Size([8, 2])


In [122]:
# Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    optimizer.zero_grad()

    outputs = model(x)

    targets = torch.randint(0, 2, (batch_size,))

    loss = criterion(outputs, targets)

    loss.backward()

    optimizer.step()

    print(epoch, loss.item())

0 0.870068371295929
1 0.5502868294715881
2 0.8339962959289551
3 0.7932228446006775
4 0.7016168832778931
5 0.7601333856582642
6 0.6520556211471558
7 0.7289938926696777
8 0.756442666053772
9 0.8044172525405884


# 8. Types of Sequence Modeling Architectures

Sequence models are categorized based on:

* number of inputs
* number of outputs
* whether data is sequential

## 1. One-to-One

Traditional neural network.

```text
Image -> Label
```

Example:

* image classification
* tabular ML
* regression
* binary classification

---

## 2. One-to-Many

A single input generates a sequence of outputs.

```text id="m7d2wp"
Single Input → Sequence Output
```

```text
Image -> Caption sequence
```

Example:

* image captioning

**Architecture Intuition**

```text id="c4n9ha"
Image Encoder → RNN Decoder
                     ↓
               word1 → word2 → word3
```

The model:

1. encodes input into a vector
2. uses that vector as initial memory
3. generates tokens one-by-one

---

## 3. Many-to-One

A sequence of inputs produces a single output.

```text id="e6k2ry"
Sequence Input → Single Output
```

```text
Word sequence -> Sentiment
```

This is one of the most common RNN architectures.

Example:

* sentiment analysis
* spam detection
* intent classification
* Electrocardiogram (ECG) classification

The RNN reads the sequence step-by-step:

```text id="r8j5mn"
x1 → h1
x2 → h2
x3 → h3
x4 → h4 → y
```

Only the **final hidden state** is used for prediction.

Above **SimpleRNN** example is exactly this type.

---

## 4. Many-to-Many (Same Length)

Input and output sequences have the SAME length.

```text id="g2m8cx"
x1 → y1
x2 → y2
x3 → y3
x4 → y4
```

```text
Sequence -> Sequence
```

Example:

* POS tagging
* named entity recognition

---

## 5. Many-to-Many (Different Length)

Input and output sequences may have different lengths.

```text id="k1x9ta"
Input Sequence → Output Sequence
```

```text
English sentence -> French sentence
```

Example:

* machine translation
* chatbots
* summarization

**Architecture Visualization**

```text id="y5m8qj"
Input Sequence
x1 → x2 → x3
          ↓
    Context Vector
          ↓
y1 → y2 → y3 → y4
```


---

## Summary Table

| Architecture                    | Input    | Output   | Example              |
| ------------------------------- | -------- | -------- | -------------------- |
| One-to-One                      | Single   | Single   | Image classification |
| One-to-Many                     | Single   | Sequence | Caption generation   |
| Many-to-One                     | Sequence | Single   | Sentiment analysis   |
| Many-to-Many (same length)      | Sequence | Sequence | POS tagging          |
| Many-to-Many (different length) | Sequence | Sequence | Translation          |